In [53]:
import argparse
import re
from pathlib import Path

import pandas as pd

In [54]:
df = pd.read_csv("data/outputs/filtered_fomo_preprocessed.csv")

used_imgs = 4
fraction_of_dataset =used_imgs / df.shape[0]

print(f"Fraction of dataset used: {fraction_of_dataset}")

# compute number of images per dataset
images_per_dataset = df["dataset"].value_counts()

# multiply the number of images per dataset by the fraction of dataset used, round to the nearest integer
images_per_dataset = (images_per_dataset * fraction_of_dataset).round().astype(int)

#  if dataset has 0 round to 1
# images_per_dataset = images_per_dataset.apply(lambda x: max(x, 1))
# print(f"Number of images per dataset: {images_per_dataset}")


# create a new csv with the number of images per dataset, take the first n images from each dataset
new_df = pd.DataFrame(columns=df.columns)
for dataset, n_images in images_per_dataset.items():
    new_df = pd.concat([new_df, df[df["dataset"] == dataset].head(n_images)])
print(f"New dataset shape: {new_df.shape}")
# images_per_dataset = new_df["dataset"].value_counts()
# print(f"Number of images per dataset: {images_per_dataset}")

# save the new csv
new_df.to_csv(f"data/outputs/filtered_fomo_preprocessed_subset_{used_imgs}.csv", index=False)

Fraction of dataset used: 0.00021266415014089
New dataset shape: (3, 13)


## Link the subset into the nnU-Net input folder

Same logic as `03_link_to_nnunet.py`, using `used_imgs` from above. Files follow the nnU-Net convention `<case_id>_0000.nii.gz` with `case_id = <dataset>_<original filename stem>`. A mapping CSV (`case_id`, `nnunet_input`, `abs_path`, `nextbrain_dseg`, ...) is saved next to the subset CSV.

In [55]:
import os

SUBSET_CSV = Path(f"data/outputs/filtered_fomo_preprocessed_subset_{used_imgs}.csv")
NNUNET_DIR = Path("/home/agustin/phd/synthseg_nextbrain/project/dev/wp4/simple_approaches/nn_unet/data/inputs/fomo")
MAPPING_CSV = SUBSET_CSV.with_name(f"{SUBSET_CSV.stem}_nnunet_mapping.csv")
OVERWRITE = False  # replace existing links


def case_id(dataset, abs_path):
    name = Path(abs_path).name
    for ext in (".nii.gz", ".nii"):
        if name.endswith(ext):
            name = name[: -len(ext)]
            break
    return f"{dataset}_{name}"


subset = pd.read_csv(SUBSET_CSV)
subset["case_id"] = [case_id(d, a) for d, a in zip(subset["dataset"], subset["abs_path"])]
dup = subset["case_id"].duplicated(keep=False)
assert not dup.any(), f"Duplicated case ids:\n{subset.loc[dup, ['case_id', 'abs_path']]}"
subset["nnunet_input"] = [str(NNUNET_DIR / f"{c}_0000.nii.gz") for c in subset["case_id"]]

NNUNET_DIR.mkdir(parents=True, exist_ok=True)
created = skipped = missing = 0
for src, dst in zip(subset["abs_path"], subset["nnunet_input"]):
    src, dst = Path(src), Path(dst)
    if not src.exists():
        print(f"[missing] {src}")
        missing += 1
        continue
    if dst.is_symlink() or dst.exists():
        if not OVERWRITE:
            skipped += 1
            continue
        dst.unlink()
    os.symlink(src, dst)
    created += 1

subset.to_csv(MAPPING_CSV, index=False)
print(f"Rows: {len(subset)} | created: {created} | already existed: {skipped} | missing source: {missing}")
print(f"Links in:    {NNUNET_DIR}")
print(f"Mapping CSV: {MAPPING_CSV}")

Rows: 3 | created: 2 | already existed: 1 | missing source: 0
Links in:    /home/agustin/phd/synthseg_nextbrain/project/dev/wp4/simple_approaches/nn_unet/data/inputs/fomo
Mapping CSV: data/outputs/filtered_fomo_preprocessed_subset_4_nnunet_mapping.csv
